In [1]:
import numpy as np
import cv2
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, GRU, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

2026-04-08 15:17:21.156201: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775661441.373125      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775661441.439965      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775661441.936116      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775661441.936161      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775661441.936164      55 computation_placer.cc:177] computation placer alr

In [3]:
def extract_frames(video_path, sequence_length=30):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    
    # Calculate how many frames to skip to get a fixed sequence length
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count / sequence_length), 1)

    for frame_counter in range(sequence_length):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        # Resize to 224x224 and normalize as per the research paper 
        resized_frame = cv2.resize(frame, (224, 224))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)
    
    video_reader.release()
    return np.array(frames_list)

In [4]:
def create_model(num_classes=5):
    # 1. Feature Extractor (ResNet50V2)
    resnet = ResNet50V2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    resnet.trainable = False  # Freeze layers to leverage prior learning 

    model = Sequential()

    # 2. Add TimeDistributed to process each frame in the sequence
    model.add(TimeDistributed(resnet, input_shape=(30, 224, 224, 3)))
    model.add(TimeDistributed(GlobalAveragePooling2D()))

    # 3. Add the GRU Layer for temporal analysis [cite: 209, 210]
    model.add(GRU(64, return_sequences=False))
    model.add(Dropout(0.4)) # Added to prevent overfitting on small datasets

    # 4. Final Classification Layer [cite: 268, 269]
    model.add(Dense(num_classes, activation='softmax'))

    return model

model = create_model()
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=['accuracy'])
model.summary()

I0000 00:00:1775661493.056627      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775661493.062494      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 30, 7, 7, 2048) │    23,564,800 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 30, 2048)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │       405,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,971,013 (91.44 MB)

 Trainable params: 406,213 (1.55 MB)

 Non-trainable params: 23,564,800 (89.89 MB)

In [7]:
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# 1. Update the path to match your actual Kaggle screenshot
# The path follows: /kaggle/input/[folder-name]/OEP database/
DATASET_DIR = "/kaggle/input/datasets/raajanwankhade/oep-dataset/OEP database"

def create_dataset():
    features = []
    labels = []
    
    # Get all subject folders (subject1, subject2, etc.)
    subject_folders = [f for f in os.listdir(DATASET_DIR) if f.startswith('subject')]
    
    for subject in subject_folders:
        subject_path = os.path.join(DATASET_DIR, subject)
        print(f'Processing: {subject}')
        
        # List all files in the subject folder
        files = os.listdir(subject_path)
        
        for file_name in files:
            # We only want video files (mp4, avi, etc.)
            if file_name.endswith(('.mp4', '.avi', '.mov')):
                video_file_path = os.path.join(subject_path, file_name)
                
                # IMPORTANT: In this dataset, labels are usually in the filename 
                # or a metadata file. Assuming the filename contains the label index (1-5).
                # Example: 'subject1_cheat1.mp4' -> Class 1
                try:
                    # Logic to extract class from filename:
                    # We look for a digit (1-5) representing the cheat type
                    # adjust this split logic based on your actual filenames!
                    class_label = int(''.join(filter(str.isdigit, file_name))[-1]) 
                    
                    if 1 <= class_label <= 5:
                        frames = extract_frames(video_file_path) # Function from Step 2
                        
                        if len(frames) == 30:
                            features.append(frames)
                            labels.append(class_label - 1) # Convert to 0-4 for encoding
                except Exception as e:
                    continue
                
    return np.array(features), to_categorical(labels, num_classes=5)

# 2. Execute and Split
X, y = create_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

Processing: subject9
Processing: subject15
Processing: subject6
Processing: subject14
Processing: subject5
Processing: subject22
Processing: subject18
Processing: subject7
Processing: subject1
Processing: subject19
Processing: subject8
Processing: subject3
Processing: subject23
Processing: subject24
Processing: subject11
Processing: subject4
Processing: subject17
Processing: subject13
Processing: subject10
Processing: subject20
Processing: subject12
Processing: subject16
Processing: subject2
Processing: subject21


In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Setup Callbacks to prevent overfitting and save the best version
early_stopping = EarlyStopping(monitor='val_loss', patience=5, mode='min', restore_best_weights=True)
checkpoint = ModelCheckpoint('cheating_detector_best.h5', monitor='val_accuracy', save_best_only=True, mode='max')

# 2. Start Training
print("Starting Training...")
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=4, # Keep batch size small for video memory
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, checkpoint]
)

Starting Training...
Epoch 1/50


I0000 00:00:1775661957.215253     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 618ms/step - accuracy: 0.0823 - loss: 2.3067

10/10 ━━━━━━━━━━━━━━━━━━━━ 221s 7s/step - accuracy: 0.0939 - loss: 2.2765 - val_accuracy: 0.6000 - val_loss: 1.0818
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 591ms/step - accuracy: 0.6567 - loss: 0.9020

10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 852ms/step - accuracy: 0.6640 - loss: 0.8883 - val_accuracy: 0.8000 - val_loss: 0.5362
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 599ms/step - accuracy: 0.9639 - loss: 0.3815

10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 855ms/step - accuracy: 0.9624 - loss: 0.3780 - val_accuracy: 0.9000 - val_loss: 0.2929
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 609ms/step - accuracy: 1.0000 - loss: 0.2339

10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 871ms/step - accuracy: 1.0000 - loss: 0.2299 - val_accuracy: 1.0000 - val_loss: 0.1698
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 834ms/step - accuracy: 0.9650 - loss: 0.1496 - val_accuracy: 1.0000 - val_loss: 0.1155
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 848ms/step - accuracy: 1.0000 - loss: 0.1169 - val_accuracy: 1.0000 - val_loss: 0.0876
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 855ms/step - accuracy: 1.0000 - loss: 0.0944 - val_accuracy: 1.0000 - val_loss: 0.0682
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 862ms/step - accuracy: 1.0000 - loss: 0.0573 - val_accuracy: 1.0000 - val_loss: 0.0561
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 849ms/step - accuracy: 1.0000 - loss: 0.0472 - val_accuracy: 1.0000 - val_loss: 0.0486
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 834ms/step - accuracy: 1.0000 - loss: 0.0357 - val_accuracy: 1.0000 - val_loss: 0.0436
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 826ms/step - accuracy: 1.0000 - loss: 0.0441 - val_accuracy: 1.0000 - va